# **Air Quality Prediction**
---
**Data Pipeline**

In [1]:
# Import the required libraries
import os

# Need to be installed
import yaml
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

## **1 - Configuration File**
---

- Create two functions: `load_config()` and `update_config()`.

In [2]:
# Function to load configuration parameter.
def load_config(path_config):
    """
    Load the configuration file.

    Parameters:
    ------
    path_config : str
        Configuration file location.

    Returns:
    ------
    params : dict
        Loaded configuration file.
    """
    
    # Try to Load config.yaml file.
    try:
        with open(path_config, 'r') as file:
            params = yaml.safe_load(file)
    except FileNotFoundError as err:
        raise RuntimeError(f'Configuration file not found in {path_config}')

    return params

In [3]:
# Function to update configuration parameter.
def update_config(key, value, params, path_config):
    """
    Update the configuration parameter values.

    Parameters:
    ------
    key : str
        The key to be updated.
    
    value : any type supported in Python
        The updated value.
    
    params : dict
        Loaded configuration parameters.
    
    path_config : str
        Configuration file location.
    
    Returns:
    ------
    config : dict
        Updated configuration parameters.
    """

    # To maintain the raw config file immutable
    params = params.copy()

    # Update the configuration
    params[key] = value

    # Write the configuration file
    with open(path_config, 'w') as file:
        yaml.dump(params, file)
    
    print(f'Params Updated! \nKey: {key} \nValue: {value}\n')

    # Reload the updated configuration file
    config = load_config(path_config)

    return config 

In [4]:
# Load the configuration file
PATH_CONFIG = '../config/config.yaml'
config = load_config(PATH_CONFIG)

In [5]:
# Check the configuration parameters.
config

{'columns_datetime': ['tanggal'],
 'columns_int': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'columns_object': ['stasiun', 'critical', 'category'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'impute_co': 11.63883089770355,
 'impute_no2': 19.330076762037685,
 'impute_o3': 32.06298655343242,
 'impute_pm10': {'BAIK': 28.426573426573427, 'TIDAK BAIK': 55.32202052091555},
 'impute_pm25': {'BAIK': 39.35454545454545, 'TIDAK BAIK': 82.33438237608182},
 'impute_so2': 35.15479651162791,
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'path_clean_test': ['../data/processed/X_test_clean.pkl',
  '../data/processed/y_test_clean.pkl'],
 'path_clean_train': ['../data/processed/X_train_clean.pkl',
  '../data/processed/y_train_clean.pkl'],
 'path_clean_valid': ['../data/processed/X_valid_clean.pkl',
  '../data/processed/y_valid_clean.pkl'],
 'path_data_joined': '../data/interim/joined_datas

## **2 - Data Collection**
---

- Create load_data() function.
- It receives one argument: data_path
- This function load all csv raw data and return the joined dataframe.

In [6]:
def load_raw_data(path_data):
    """
    Load csv files and join them into one dataframe.

    Parameters:
    ----------
    path_data : str
        Raw dataset location.
    
    Returns:
    -------
    raw_dataset : pd.DataFrame
        Loaded and joined data.
    """

    # Create dataframe
    raw_dataset = pd.DataFrame()

    # Load and join the csv files
    for i in os.listdir(path_data):
        raw_dataset = pd.concat([
            pd.read_csv(path_data + i), raw_dataset
        ])

    return raw_dataset

In [7]:
# Load the raw dataset
PATH_DATA_RAW = '../data/raw/'
raw_dataset = load_raw_data(PATH_DATA_RAW)

In [8]:
# Update the configuration parameter
config = update_config(
    key = 'path_data_raw',
    value = PATH_DATA_RAW,
    params = config,
    path_config = PATH_CONFIG
)

Params Updated! 
Key: path_data_raw 
Value: ../data/raw/



In [9]:
# Check the raw dataset
raw_dataset

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
0,2021-09-01,DKI1 (Bunderan HI),63,88,29,15,24,38,88,PM25,SEDANG
1,2021-09-02,DKI1 (Bunderan HI),60,83,29,11,30,28,83,PM25,SEDANG
2,2021-09-03,DKI1 (Bunderan HI),60,82,27,11,37,30,82,PM25,SEDANG
3,2021-09-04,DKI1 (Bunderan HI),58,77,26,10,31,28,77,PM25,SEDANG
4,2021-09-05,DKI1 (Bunderan HI),63,85,27,11,28,28,85,PM25,SEDANG
...,...,...,...,...,...,...,...,...,...,...,...
150,2021-08-27,DKI5 (Kebon Jeruk) Jakarta Barat,61,96,34,8,29,15,96,PM25,SEDANG
151,2021-08-28,DKI5 (Kebon Jeruk) Jakarta Barat,63,100,31,8,44,12,100,PM25,SEDANG
152,2021-08-29,DKI5 (Kebon Jeruk) Jakarta Barat,67,111,32,10,36,13,111,PM25,TIDAK SEHAT
153,2021-08-30,DKI5 (Kebon Jeruk) Jakarta Barat,83,126,35,16,32,29,126,PM25,TIDAK SEHAT


- We found that:
    1. Index only ranged from 0 to 154, while there are 1830 rows.

In [10]:
# Try to reset the index to solve the first problem
raw_dataset = raw_dataset.reset_index(drop=True)

# Check the updated index
raw_dataset

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
0,2021-09-01,DKI1 (Bunderan HI),63,88,29,15,24,38,88,PM25,SEDANG
1,2021-09-02,DKI1 (Bunderan HI),60,83,29,11,30,28,83,PM25,SEDANG
2,2021-09-03,DKI1 (Bunderan HI),60,82,27,11,37,30,82,PM25,SEDANG
3,2021-09-04,DKI1 (Bunderan HI),58,77,26,10,31,28,77,PM25,SEDANG
4,2021-09-05,DKI1 (Bunderan HI),63,85,27,11,28,28,85,PM25,SEDANG
...,...,...,...,...,...,...,...,...,...,...,...
1825,2021-08-27,DKI5 (Kebon Jeruk) Jakarta Barat,61,96,34,8,29,15,96,PM25,SEDANG
1826,2021-08-28,DKI5 (Kebon Jeruk) Jakarta Barat,63,100,31,8,44,12,100,PM25,SEDANG
1827,2021-08-29,DKI5 (Kebon Jeruk) Jakarta Barat,67,111,32,10,36,13,111,PM25,TIDAK SEHAT
1828,2021-08-30,DKI5 (Kebon Jeruk) Jakarta Barat,83,126,35,16,32,29,126,PM25,TIDAK SEHAT


- Now the index problem are fixed.

In [11]:
# Serialize the joined dataset
PATH_DATA_JOINED = '../data/interim/joined_dataset.pkl'
joblib.dump(raw_dataset, PATH_DATA_JOINED)

['../data/interim/joined_dataset.pkl']

In [12]:
# Update the configuration parameter.
config = update_config(
    key = 'path_data_joined',
    value = PATH_DATA_JOINED,
    params = config,
    path_config = PATH_CONFIG
)

Params Updated! 
Key: path_data_joined 
Value: ../data/interim/joined_dataset.pkl



## **3 - Data Validation**
---

In [13]:
# Check the data type for each feature
raw_dataset.dtypes

tanggal        str
stasiun        str
pm10        object
pm25        object
so2         object
co          object
o3          object
no2         object
max         object
critical       str
categori       str
dtype: object

- Several features don't have the same configuration data type.
- We need to handle those error columns.

### 3.1. Handling Column `tanggal`

In [14]:
# Try to cast the column to datetime type
raw_dataset['tanggal'] = pd.to_datetime(raw_dataset['tanggal'])

In [15]:
raw_dataset.dtypes

tanggal     datetime64[us]
stasiun                str
pm10                object
pm25                object
so2                 object
co                  object
o3                  object
no2                 object
max                 object
critical               str
categori               str
dtype: object

### 3.2. Handling Column `pm10`

In [16]:
raw_dataset['pm10'] = raw_dataset['pm10'].astype(int)

ValueError: invalid literal for int() with base 10: '---'

- `ValueError` occurs, it tells us that there are data that isn't integer (`"---"`).
- We will replace those `"---"` with a value that doesn't exist in the column.
- Based on data definition, we know that we can use `-1`.

In [17]:
# Ensure no single data that is -1
raw_dataset.eq("-1").any() | raw_dataset.eq(-1).any()

tanggal     False
stasiun     False
pm10        False
pm25        False
so2         False
co          False
o3          False
no2         False
max         False
critical    False
categori    False
dtype: bool

In [18]:
# Replace the "---" with -1 and cast the column into int.
raw_dataset['pm10'] = raw_dataset['pm10'].replace('---',-1).astype(int)

### 3.3. Handling Column `pm25`

In [19]:
# Try to cast the column to int type
raw_dataset['pm25'] = raw_dataset['pm25'].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [20]:
# Replace the "---" with -1 and cast the column into int
raw_dataset['pm25'] = raw_dataset['pm25'].replace('---',-1).astype(int)

ValueError: cannot convert float NaN to integer

- There is a different `ValueError`.
- There are `NaN` values, thus we can't directly convert to int.
- We need to handle this problem first.

In [21]:
# Sanity check the missing values
raw_dataset['pm25'].isnull().sum()

np.int64(62)

- There are 62 `NaN` values. For now, we can replace it with `-1`.

In [22]:
# Replace the NaN values with -1
raw_dataset['pm25'] = raw_dataset['pm25'].fillna(-1)

# Sanity check the missing values
raw_dataset['pm25'].isnull().sum()

np.int64(0)

In [23]:
# Replace the "---" with -1 and cast the column into int
raw_dataset['pm25'] = raw_dataset['pm25'].replace('---',-1).astype(int)

### 3.4. Handling Column `so2`

In [24]:
# Try to cast the column to int type
raw_dataset['so2'] = raw_dataset['so2'].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [25]:
# Replace the "---" with -1 and cast the column into int
raw_dataset['so2'] = raw_dataset['so2'].replace('---',-1).astype(int)

### 3.5. Handling Column `co`

In [26]:
# Try to cast the column to int type
raw_dataset['co'] = raw_dataset['co'].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [27]:
# Replace the "---" with -1 and cast the column into int
raw_dataset['co'] = raw_dataset['co'].replace('---',-1).astype(int)

### 3.6. Handling Column `o3`

In [28]:
# Try to cast the column to int type
raw_dataset['o3'] = raw_dataset['o3'].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [29]:
# Replace the "---" with -1 and cast the column into int
raw_dataset['o3'] = raw_dataset['o3'].replace('---',-1).astype(int)

### 3.7. Handling Column `no2`

In [30]:
# Try to cast the column to int type
raw_dataset['no2'] = raw_dataset['no2'].astype(int)

ValueError: invalid literal for int() with base 10: '---'

In [31]:
# Replace the "---" with -1 and cast the column into int
raw_dataset['no2'] = raw_dataset['no2'].replace('---',-1).astype(int)

### 3.8. Handling Column `max`

In [32]:
# Try to cast the column to int type
raw_dataset['max'] = raw_dataset['max'].astype(int)

ValueError: invalid literal for int() with base 10: 'PM25'

- Seems like the error is different.
- There is data with value `"PM25"` in the `max` column.
- We need to investigate the data.

In [33]:
# Check which data that cause error
raw_dataset[raw_dataset['max'] == 'PM25']

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
307,2021-12-03,DKI1 (Bunderan HI),49,31,9,19,7,49,PM25,BAIK,NaN


- Looks like there are typos on row index 762.
    - The `"BAIK"` value must be on `categori` column.
    - We need to investigate what do `max` and `critical` column represent.
- Let's randomly sample 5 data.

In [34]:
raw_dataset.sample(5)

,tanggal,stasiun,pm10,pm25,so2,co,o3,no2,max,critical,categori
733,2021-11-29,DKI4 (Lubang Buaya),29,55,37,8,17,11,55,PM25,SEDANG
1365,2021-01-01,DKI1 (Bunderan HI),38,53,29,6,31,13,53,PM25,SEDANG
1682,2021-08-08,DKI1 (Bunderan HI),53,69,33,6,23,16,69,PM25,SEDANG
1472,2021-01-15,DKI4 (Lubang Buaya),51,-1,26,20,56,5,56,CO,SEDANG
1478,2021-01-21,DKI4 (Lubang Buaya),63,-1,25,21,38,7,63,PM10,SEDANG


- Looks like the `max` column represents the maximum value between any other int columns.
- And looks like the `critical` column represents the column name of `max` value.

- Thus, let's fix the error on the row index 762:
    - Replace the `max` column with `pm10` or `no2` value, let's take from `pm10`.
    - Replace the `critical` column with `"PM10"`.
    - Replace the `categori` column with `"BAIK"`.

In [35]:
# Fix the error
error_index = raw_dataset[raw_dataset['max'] == 'PM25'].index[0]
raw_dataset.loc[error_index, 'max'] = raw_dataset.loc[error_index, 'pm10']
raw_dataset.loc[error_index, 'critical'] = 'PM10'
raw_dataset.loc[error_index, 'categori'] = 'BAIK'

In [36]:
# Sanity check the result
raw_dataset.loc[error_index]

tanggal     2021-12-03 00:00:00
stasiun      DKI1 (Bunderan HI)
pm10                         49
pm25                         31
so2                           9
co                           19
o3                            7
no2                          49
max                          49
critical                   PM10
categori                   BAIK
Name: 307, dtype: object

In [37]:
raw_dataset['max'] = raw_dataset['max'].astype(int)

### 3.9. Handling Column `critical`

In [38]:
# Check the unique value
raw_dataset['critical'].value_counts()

critical
PM25    1631
PM10      65
O3        57
CO        34
SO2       26
Name: count, dtype: int64

- Seems like no action needed.

### 3.10. Handling Column `categori`

In [39]:
# Check the unique value
raw_dataset['categori'].value_counts()

categori
SEDANG            1305
TIDAK SEHAT        319
BAIK               189
TIDAK ADA DATA      17
Name: count, dtype: int64

- There are 17 `"TIDAK ADA DATA"` values, indicating the missing label.
- Since we don't know which label that can replace the `"TIDAK ADA DATA"`, thus we can drop those data.

In [40]:
# Drop the "TIDAK ADA DATA" category
missing_labels = raw_dataset[raw_dataset['categori'] == 'TIDAK ADA DATA']
raw_dataset = raw_dataset.drop(index = missing_labels.index)

In [41]:
# Sanity check the result
raw_dataset['categori'].value_counts()

categori
SEDANG         1305
TIDAK SEHAT     319
BAIK            189
Name: count, dtype: int64

- Let's rename the `categori` column into the proper name, `category`.
- Don't forget to update the configuration file.

In [42]:
# Rename "categori" into "category"
raw_dataset = raw_dataset.rename(columns = {'categori': 'category'})

In [43]:
# Update the configuration parameter
config = update_config(
    key = 'label',
    value = 'category',
    params = config,
    path_config = PATH_CONFIG
)

Params Updated! 
Key: label 
Value: category



In [44]:
# Update the configuration parameter
col_object = config['columns_object']
col_object[-1] = 'category'

config = update_config(
    key = 'columns_object',
    value = col_object,
    params = config,
    path_config = PATH_CONFIG
)

Params Updated! 
Key: columns_object 
Value: ['stasiun', 'critical', 'category']



In [45]:
# Sanity check the data types
raw_dataset.info()

<class 'pandas.DataFrame'>
Index: 1813 entries, 0 to 1829
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   tanggal   1813 non-null   datetime64[us]
 1   stasiun   1813 non-null   str           
 2   pm10      1813 non-null   int64         
 3   pm25      1813 non-null   int64         
 4   so2       1813 non-null   int64         
 5   co        1813 non-null   int64         
 6   o3        1813 non-null   int64         
 7   no2       1813 non-null   int64         
 8   max       1813 non-null   int64         
 9   critical  1813 non-null   str           
 10  category  1813 non-null   str           
dtypes: datetime64[us](1), int64(7), str(3)
memory usage: 170.0 KB


- All columns are already the same as in the data definition.
- Now serialize the validated data.

In [46]:
# Serialize the validated data
PATH_DATA_VALIDATED = '../data/interim/validated_data.pkl'
joblib.dump(raw_dataset, PATH_DATA_VALIDATED)

['../data/interim/validated_data.pkl']

In [47]:
# Update the configuration parameter
config = update_config(
    key = 'path_data_validated',
    value = PATH_DATA_VALIDATED,
    params = config,
    path_config = PATH_CONFIG
)

Params Updated! 
Key: path_data_validated 
Value: ../data/interim/validated_data.pkl



## **4 - Update the Range of Data in Configuration File**
---

In [48]:
# Update the range of data with the min and max value of each column
cols = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
param_keys = ['range_pm10', 'range_pm25', 'range_so2', 'range_co', 'range_o3', 'range_no2']

for col, key in zip(cols, param_keys):
    config = update_config(
        key = key,
        value = [int(np.min(raw_dataset[col])), int(np.max(raw_dataset[col]))],
        params = config,
        path_config = PATH_CONFIG
    )

Params Updated! 
Key: range_pm10 
Value: [-1, 179]

Params Updated! 
Key: range_pm25 
Value: [-1, 174]

Params Updated! 
Key: range_so2 
Value: [-1, 82]

Params Updated! 
Key: range_co 
Value: [-1, 47]

Params Updated! 
Key: range_o3 
Value: [-1, 151]

Params Updated! 
Key: range_no2 
Value: [-1, 65]



## **5 - Data Defense**
---

- Create the `check_data()` function.
- It receives 2 arguments: `input_data` and `params`.
    - `input_data` is the raw dataset.
    - `params` is the configuration parameters.
- It is a void function (no return value).
- If `AssertionError` happens, there are existing data that don't match the configuration.

In [49]:
# Function for data defense
def data_defense(data, config):
    """
    Do data defense for checking the data types and range of data.

    Parameters:
    ----------
    data : pd.DataFrame
        The data to be checked.
    
    config : dict
        Loaded configuration parameters.
    
    Returns:
    -------
    None, it's a void function.
    """

    # Check data types
    assert data.select_dtypes('datetime').columns.to_list() == config['columns_datetime'], "an error occurs in datetime column(s)."
    assert data.select_dtypes('object').columns.to_list() == config['columns_object'], "an error occurs in object column(s)."
    assert data.select_dtypes('int').columns.to_list() == config['columns_int'], "an error occurs in int column(s)."

    # Check range of data
    assert set(data['stasiun']).issubset(set(config['range_stasiun'])), "an error occurs in stasiun range."
    assert data['pm10'].between(config['range_pm10'][0], config['range_pm10'][1]).sum() == len(data), "an error occurs in pm10 range."
    assert data['pm25'].between(config['range_pm25'][0], config['range_pm25'][1]).sum() == len(data), "an error occurs in pm25 range."
    assert data['so2'].between(config['range_so2'][0], config['range_so2'][1]).sum() == len(data), "an error occurs in so2 range."
    assert data['co'].between(config['range_co'][0], config['range_co'][1]).sum() == len(data), "an error occurs in co range."
    assert data['o3'].between(config['range_o3'][0], config['range_o3'][1]).sum() == len(data), "an error occurs in o3 range."
    assert data['no2'].between(config['range_no2'][0], config['range_no2'][1]).sum() == len(data), "an error occurs in no2 range."



In [50]:
# Do data defense
data_defense(raw_dataset, config)

/tmp/ipykernel_197955/4138669405.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  assert data.select_dtypes('object').columns.to_list() == config['columns_object'], "an error occurs in object column(s)."


- Seems like our data are in good condition!

## **6 - Data Split**
---

In [51]:
# Function for Input-Output Split
def split_input_output(data, config):
    """
    Split the input(X) and output (y).

    Parameters:
    ----------
    data : pd.DataFrame
        The processed dataset.
    
    config : dict
        Loaded configuration parameters.
    
    Returns:
    -------
    X : pd.DataFrame
        The input data.
    
    y : pd.DataFrame
        The output data.
    """

    # Ensure raw data immutable.
    data = data.copy()

    # Split the X and y
    X = data[config['features']]
    y = data[config['label']]

    print(f'Original data shape : {data.shape}')
    print(f'Selected Features   : {config['features']}')
    print(f'X data shape        : {X.shape}')
    print(f'y data shape        : {y.shape}')

    return X, y

In [52]:
# Split input-output
X, y = split_input_output(
    data = raw_dataset,
    config = config
)

Original data shape : (1813, 11)
Selected Features   : ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
X data shape        : (1813, 7)
y data shape        : (1813,)


In [53]:
# Function for Train-Test Split.
def split_train_test(X, y, test_size, random_state=123):
    """
    Split the train and test set.

    Parameters:
    ----------
    X : pd.DataFrame
        The input data.
    
    y : pd.Series
        The output data.
    
    test_size : float
        The proportion of test set.
    
    random_state : int, default = 123
        For reproducibility.
    
    Returns:
    -------
    X_train, X_test : pd.DataFrame
        The train and test input.
    
    y_train, y_test : pd.Series
        The train and test output.
    """
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size = test_size,
        random_state = random_state,
        stratify = y
    )

    print(f'X_train shape : {X_train.shape}')
    print(f'y_train shape : {y_train.shape}')
    print(f'X_test shape  : {X_test.shape}')
    print(f'y_test shape  : {y_test.shape}\n')

    return X_train, X_test, y_train, y_test

In [54]:
# train:valid:test -> 80:10:10

# Train vs not-train
X_train, X_not_train, y_train, y_not_train = split_train_test(
    X = X,
    y = y,
    test_size = 0.2
)

# Valid vs test
X_valid, X_test, y_valid, y_test = split_train_test(
    X = X_not_train,
    y = y_not_train,
    test_size = 0.5
)

X_train shape : (1450, 7)
y_train shape : (1450,)
X_test shape  : (363, 7)
y_test shape  : (363,)

X_train shape : (181, 7)
y_train shape : (181,)
X_test shape  : (182, 7)
y_test shape  : (182,)



In [55]:
# Sanity check the train data
data_train = pd.concat(
    [X_train, y_train],
    axis = 1
)

data_train.sample(10)

,stasiun,pm10,pm25,so2,co,o3,no2,category
686,DKI3 (Jagakarsa),44,61,47,10,30,12,SEDANG
406,DKI4 (Lubang Buaya),38,63,40,9,28,19,SEDANG
672,DKI2 (Kelapa Gading),18,-1,3,7,47,9,BAIK
700,DKI3 (Jagakarsa),51,71,41,17,-1,10,SEDANG
207,DKI2 (Kelapa Gading),62,83,49,12,63,19,SEDANG
643,DKI1 (Bunderan HI),23,28,30,7,12,5,BAIK
1388,DKI1 (Bunderan HI),33,48,12,14,23,10,BAIK
317,DKI1 (Bunderan HI),31,46,34,10,13,6,BAIK
449,DKI5 (Kebon Jeruk) Jakarta Barat,47,73,20,15,31,49,SEDANG
581,DKI4 (Lubang Buaya),51,79,41,15,45,15,SEDANG


## **7 - Data Serialization**
---

In [56]:
# Serialize the splitted data
PATH_DATA_SPLITTED = '../data/interim/'

data_configuration = {
    'train': {
        'X_train': X_train,
        'y_train': y_train
    },
    'valid': {
        'X_valid': X_valid,
        'y_valid': y_valid
    },
    'test': {
        'X_test': X_test,
        'y_test': y_test
    }
}

for key, value in data_configuration.items():
    config_key = f'path_data_{key}'
    config_value = []

    for v in value:
        # Get each path
        path = f'{PATH_DATA_SPLITTED + v}.pkl'
        config_value.append(path)

        # Get each data
        data = value[v]

        # Serialize the splitted data
        joblib.dump(data, path)

    # Update the configuration parameters
    config = update_config(
        key = config_key,
        value = config_value,
        params = config,
        path_config = PATH_CONFIG
    )

Params Updated! 
Key: path_data_train 
Value: ['../data/interim/X_train.pkl', '../data/interim/y_train.pkl']

Params Updated! 
Key: path_data_valid 
Value: ['../data/interim/X_valid.pkl', '../data/interim/y_valid.pkl']

Params Updated! 
Key: path_data_test 
Value: ['../data/interim/X_test.pkl', '../data/interim/y_test.pkl']



In [57]:
# Check the configuration parameters
config

{'columns_datetime': ['tanggal'],
 'columns_int': ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2', 'max'],
 'columns_object': ['stasiun', 'critical', 'category'],
 'features': ['stasiun', 'pm10', 'pm25', 'so2', 'co', 'o3', 'no2'],
 'impute_co': 11.63883089770355,
 'impute_no2': 19.330076762037685,
 'impute_o3': 32.06298655343242,
 'impute_pm10': {'BAIK': 28.426573426573427, 'TIDAK BAIK': 55.32202052091555},
 'impute_pm25': {'BAIK': 39.35454545454545, 'TIDAK BAIK': 82.33438237608182},
 'impute_so2': 35.15479651162791,
 'label': 'category',
 'label_categories': ['BAIK', 'SEDANG', 'TIDAK SEHAT'],
 'label_categories_new': ['BAIK', 'TIDAK BAIK'],
 'path_clean_test': ['../data/processed/X_test_clean.pkl',
  '../data/processed/y_test_clean.pkl'],
 'path_clean_train': ['../data/processed/X_train_clean.pkl',
  '../data/processed/y_train_clean.pkl'],
 'path_clean_valid': ['../data/processed/X_valid_clean.pkl',
  '../data/processed/y_valid_clean.pkl'],
 'path_data_joined': '../data/interim/joined_datas